<a href="https://colab.research.google.com/github/varunrao0606-dev/Rapid_Metro/blob/main/GGN_Rapid_Metro_Spatial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Gurugram Rapid Metro- Spatial Data Preparation
# Ward areas and OSM metro network extraction

import geopandas as gpd
import pandas as pd
import osmnx as ox


# Ward areas

gdf = gpd.read_file("wards_gurugram.geojson")

# EPSG:32643 = UTM zone for Delhi/Gurgaon, needed to measure in metres
gdf_projected = gdf.to_crs(epsg=32643)
gdf["Area_Sq_Km"] = gdf_projected.geometry.area / 1_000_000

drop_cols = ["tessellate", "extrude", "visibility", "geometry"]
keep_cols = [c for c in gdf.columns if c not in drop_cols]

pd.set_option("display.max_rows", None)
print(gdf[keep_cols + ["Area_Sq_Km"]])


# Metro network extraction (OSM)

place = "Gurugram, Haryana, India"

# Corridors
metro_lines = ox.features_from_place(place, tags={"railway": ["subway", "light_rail"]})
metro_lines = metro_lines[metro_lines.geometry.type == "LineString"]
metro_lines.to_file("Gurugram_Metro_Lines.geojson", driver="GeoJSON")

# Stations
stations = ox.features_from_place(place, tags={"railway": "station"})
stations = stations[stations.geometry.type == "Point"]
stations.to_file("Gurugram_Metro_Stations.geojson", driver="GeoJSON")

# 800m catchment buffers, projected to metres and back to WGS84
stations_proj = stations.to_crs(epsg=32643)
catchment_buffers = stations_proj.copy()
catchment_buffers["geometry"] = stations_proj.geometry.buffer(800)
catchment_buffers.to_crs(epsg=4326).to_file("Station_800m_Catchment.geojson", driver="GeoJSON")